<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/A5_Multi_Step_Forecasting.ipynb)

# Tomorrow, Three Ways: Recursive, Direct, Multi-Output
--------------------------------------------------
**Dr. Dave Wanik - University of Connecticut**

The capstone forecast 24 hours at once with `Dense(24)`. That is one of **three** ways to get from a one-step model to a whole day, and the one most people try first is the one with a trap in it:

| strategy | how | the catch |
| :-- | :-- | :-- |
| **Recursive** (autoregressive) | one model, one hour ahead; feed the prediction back in as "last hour" and roll forward 24 times | your own errors become tomorrow's inputs - and the covariates at future hours have to come from *somewhere* |
| **Direct** | one model per horizon: a 1-hour model, a 2-hour model, ... a 24-hour model | 24 models to train and maintain; nothing ties the horizons together |
| **Multi-output** (MIMO) | one model, `Dense(24)` - the capstone | one shot, no rolling; the model has to learn the whole daily shape at once |

You'll build all three on the same data and plot **error by horizon** for each. Then the question that trips everyone up in practice: when past demand is a feature, *what goes in that column for the hours you haven't seen yet?*

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 18 — Tomorrow, three ways: recursive, direct, multi-output
- Setup: the capstone's multivariate one-step LSTM (last 24 hours -> next hour, demand's own past in the window). The question: how do you get 24 hours out of it?
- RECURSIVE: predict hour 1, write it into the demand column, slide the window, predict hour 2... Draw the window sliding on screen. The trap: after a few steps the window is mostly your own guesses.
- The covariate problem is the point of the video: for the future rows you need the clock (known - free), the weather (NOT known - use a forecast; here we cheat with the actuals and CALL it perfect foresight, then show the honest version with the weather frozen), and past demand (your own predictions).
- Exposure bias: the model trained on TRUE past demand, then runs on its own predictions. Errors compound - read the curve: 33 MW at hour 1 (best of everything), 331 at hour 24 with perfect weather, 414 with the weather frozen - WORSE than seasonal naive (227) from about hour 8 on.
- DIRECT: retrain the same model with the target shifted h hours out - a separate model per horizon (we do 1, 6, 12, 24: 42 / 142 / 178 / 207 MW). No rolling, no fake inputs; the cost is 24 models.
- MULTI-OUTPUT: Dense(24), the capstone. One model, one shot: 87 / 139 / 171 / 206 MW - weak at hour 1 (it's learning the whole day at once), tied with direct by hour 24, and both beat seasonal naive. Say which you'd ship and why (direct/MIMO for ops; recursive only for the first few hours).
- Real life: the weather column at future hours comes from the NWS forecast (we have a pipeline for that in class) - and the forecast's error becomes yours.
-->


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from keras.models import Sequential, load_model
from keras.layers import Input, LSTM, Dense
from keras.callbacks import EarlyStopping
import keras
keras.utils.set_random_seed(5509)   # reproducibility: same numbers every run (CPU exact; a GPU may drift a little)

The three strategies, before any code. They differ in exactly one way: **what the model is fed when it
reaches an hour that has not happened yet.**

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/Module4/img/m4_multistep_strategies.png" width="100%" alt="recursive vs direct vs multi-output"/>

Keep the picture in mind as each one is built below. The thing to watch for is the dashed red loop in strategy 1 -
that loop is the only place a model is ever fed its own guess, and it is the whole reason its error grows faster
than the other two.

## Data and the one-step engine

Same prep as the capstone. The one-step LSTM is the engine every strategy starts from.

<!-- WINDOW-FN -->
## The window-making function: `split_sequences`

Target in the **last column**, then:

- **`n_steps`** is the **look-back**: X is the last `n_steps` rows of **every column, the target included**. Yesterday's demand is the most useful input there is.
- y is the target **`ahead` steps (1 = the very next step)** after the window ends.

Result: X is `(samples, n_steps, n_features)`.


In [ ]:
# same prep as the demand capstone: sort, fill, clock features, Demand LAST, train 2017-18 / test 2019
url = "https://raw.githubusercontent.com/drdave-teaching/OPIM5509Files/main/OPIM5509_Module4_Files/data/BDL_cleanweather_energy.csv"
df = pd.read_csv(url, parse_dates=["Datetime"]).sort_values("Datetime").set_index("Datetime").ffill()

def add_clock(d):
    d = d.copy()
    d["hour_sin"] = np.sin(2*np.pi*d.index.hour/24);      d["hour_cos"] = np.cos(2*np.pi*d.index.hour/24)
    d["dow_sin"]  = np.sin(2*np.pi*d.index.dayofweek/7);  d["dow_cos"]  = np.cos(2*np.pi*d.index.dayofweek/7)
    return d

feats = ["BDL_tmpf", "BDL_dwpf", "BDL_relh", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "Demand"]   # Demand LAST
data  = add_clock(df.loc["2017-01-01":"2019-12-31"])[feats]
train, test = data.loc[:"2018-12-31"], data.loc["2019-01-01":]

sc   = MinMaxScaler().fit(train)               # fit on TRAIN only
sc_y = MinMaxScaler().fit(train[["Demand"]])   # target-only scaler to get MW back
tr_s, te_s = sc.transform(train), sc.transform(test)

def split_sequences(seqs, n_steps, ahead=1):
    """inputs = the past n_steps rows (every column); target = Demand `ahead` steps after the window ends"""
    X, y = [], []
    for i in range(len(seqs) - n_steps - ahead + 1):
        X.append(seqs[i:i+n_steps, :]); y.append(seqs[i+n_steps+ahead-1, -1])
    return np.array(X), np.array(y)

n_steps = 24
X_train, y_train = split_sequences(tr_s, n_steps)
X_test,  y_test  = split_sequences(te_s, n_steps)
n_features = X_train.shape[2]
y_true = sc_y.inverse_transform(y_test.reshape(-1, 1)).ravel()
es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=0)
print("train:", X_train.shape, "| test:", X_test.shape)

In [ ]:
one_step = Sequential()
one_step.add(Input(shape=(n_steps, n_features)))
one_step.add(LSTM(32))
one_step.add(Dense(1))
one_step.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = one_step.fit(X_train, y_train, epochs=50, batch_size=64, validation_split=0.2, callbacks=[es], verbose=0)
# the loss curve - always keep the History object that fit() hands back
plt.figure(figsize=(7, 2.8))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.xlabel("epoch"); plt.ylabel("loss (mse)"); plt.title("the one-step engine: loss curve")
plt.legend(); plt.tight_layout(); plt.show()
print("stopped after", len(history.history["loss"]), "epochs")

pred_1h = sc_y.inverse_transform(one_step.predict(X_test, verbose=0)).ravel()
print(f"one-step LSTM, 1 hour ahead: MAE {mean_absolute_error(y_true, pred_1h):.1f} MW")

## Strategy 1: recursive (autoregressive)

Take the last 24 real hours, predict hour +1. Now build the row for hour +1: the **clock** we know, the **weather** we... don't (more on that below), and **demand** is the number we just predicted. Slide the window forward one row and predict hour +2. Repeat 24 times.

The model was trained where the demand column was always *true* history. At forecast time it sees its own guesses in that column - that mismatch has a name, **exposure bias** - and every error it makes becomes part of tomorrow's input.

In [ ]:
def recursive_forecast(model, seqs, n_steps, horizon, weather_cols, mode="actual"):
    """Roll a one-step model horizon steps ahead from EVERY window in seqs.
    mode='actual'  -> future weather rows come from the real future (perfect foresight - a cheat, labelled as such)
    mode='persist' -> future weather is frozen at the last observed hour (what you'd have with no forecast)"""
    n_orig = len(seqs) - n_steps - horizon + 1

    # every origin's starting window: 24 real hours of history
    window = []
    for i in range(n_orig):
        window.append(seqs[i:i+n_steps])
    window = np.array(window)                                            # (origins, 24, 8)

    preds = np.zeros((n_orig, horizon))
    for h in range(horizon):
        p = model.predict(window, verbose=0).ravel()                     # the next hour, for every origin at once
        preds[:, h] = p

        # the real row for hour +h+1. We only keep its CLOCK columns.
        future = []
        for i in range(n_orig):
            future.append(seqs[i+n_steps+h])
        future = np.array(future).copy()
        if mode == "persist":
            future[:, weather_cols] = window[:, -1, weather_cols]            # no forecast: freeze the weather
        future[:, -1] = p                                                    # the demand column is OUR PREDICTION now
        window = np.concatenate([window[:, 1:, :], future[:, None, :]], axis=1)   # slide the window
    return preds

horizon = 24
weather_cols = [feats.index(c) for c in ["BDL_tmpf", "BDL_dwpf", "BDL_relh"]]
rec_actual  = recursive_forecast(one_step, te_s, n_steps, horizon, weather_cols, mode="actual")
rec_persist = recursive_forecast(one_step, te_s, n_steps, horizon, weather_cols, mode="persist")

# the truth for the same origins and horizons
n_orig = rec_actual.shape[0]
truth = []
for i in range(n_orig):
    truth.append(te_s[i+n_steps:i+n_steps+horizon, -1])
truth = np.array(truth)

def to_mw(a):
    """Undo the target scaler, keeping the (origins, horizon) shape."""
    flat = sc_y.inverse_transform(a.reshape(-1, 1))
    return flat.reshape(a.shape)

truth_mw       = to_mw(truth)
rec_actual_mw  = to_mw(rec_actual)
rec_persist_mw = to_mw(rec_persist)

mae_rec_actual  = np.abs(rec_actual_mw  - truth_mw).mean(axis=0)
mae_rec_persist = np.abs(rec_persist_mw - truth_mw).mean(axis=0)
print("recursive, perfect-foresight weather - MAE at h=1, 6, 12, 24:", mae_rec_actual[[0, 5, 11, 23]].round(1))
print("recursive, weather frozen          - MAE at h=1, 6, 12, 24:", mae_rec_persist[[0, 5, 11, 23]].round(1))

## Strategy 2: direct (one model per horizon)

Same window, but the target is demand **h hours after the window ends**. Nothing is fed back, so there is no exposure bias - the model at horizon 12 learned, from real data, what 12 hours of drift looks like. The price: one model per horizon. We train four (1, 6, 12, 24) to see the curve; production would train all 24.

In [ ]:
direct_h, direct_mae = [1, 6, 12, 24], []
direct_hist = {}
for h in direct_h:
    Xh_tr, yh_tr = split_sequences(tr_s, n_steps, ahead=h)
    Xh_te, yh_te = split_sequences(te_s, n_steps, ahead=h)
    m = Sequential()
    m.add(Input(shape=(n_steps, n_features)))
    m.add(LSTM(32))
    m.add(Dense(1))
    m.compile(optimizer="adam", loss="mse")
    history = m.fit(Xh_tr, yh_tr, epochs=25, batch_size=64, validation_split=0.2, callbacks=[es], verbose=0)
    direct_hist[h] = history

    # score on the same origins as the recursive run (first n_orig windows)
    p = sc_y.inverse_transform(m.predict(Xh_te[:n_orig], verbose=0)).ravel()
    t = sc_y.inverse_transform(yh_te[:n_orig].reshape(-1, 1)).ravel()
    direct_mae.append(mean_absolute_error(t, p))
    print(f"direct model, {h:2d} hours ahead: MAE {direct_mae[-1]:.1f} MW")

# one curve per horizon - four models, four fits
plt.figure(figsize=(7, 2.8))
for h in direct_h:
    plt.plot(direct_hist[h].history["val_loss"], label="trained for t+" + str(h))
plt.xlabel("epoch"); plt.ylabel("validation loss (mse)")
plt.title("direct: one model per horizon")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()


## Strategy 3: multi-output (`Dense(24)`)

The capstone's approach: one model, 24 outputs, no rolling. Same 24-hour window so the comparison is fair.

<!-- WINDOW-FN -->
## The window-making function: `split_multistep`

For forecasting **several steps at once**:

- **`lookback`**: how many past rows go in (X is `(samples, lookback, n_features)`).
- **`horizon`**: how many future values come out (y is `(samples, horizon)`), one output per step, so the model ends in `Dense(horizon)`.

With `lookback=168, horizon=24`: the past week in, the next day out.


In [ ]:
def split_multistep(seqs, lookback, horizon):
    X, y = [], []
    for i in range(len(seqs) - lookback - horizon + 1):
        X.append(seqs[i:i+lookback, :]); y.append(seqs[i+lookback:i+lookback+horizon, -1])
    return np.array(X), np.array(y)

Xm_tr, ym_tr = split_multistep(tr_s, n_steps, horizon)
Xm_te, ym_te = split_multistep(te_s, n_steps, horizon)


### What exactly is one sample?

> **One sample is 24 hours of all 8 columns, and the label is 24 numbers - demand for each hour of the next day.**

Compare it with the one-step engine at the top of this notebook: identical X, and `y` goes from one number to
twenty-four. That is the entire difference between strategy 3 and everything before it.

In [ ]:
print("X:", Xm_tr.shape, "= (samples, 24 hours, 8 features)  <- same as the one-step model")
print("y:", ym_tr.shape, "= (samples, 24)                  <- 24 answers instead of 1")
print()
print("one sample - Xm_tr[0] - is a 24 x 8 block. Its last 3 hours:")
display(pd.DataFrame(Xm_tr[0][-3:], columns=feats,
                     index=["t-2", "t-1", "t"]).round(3))
print()
print("its label ym_tr[0], the next 24 hours of demand, in MW:")
print(sc_y.inverse_transform(ym_tr[0].reshape(-1, 1)).ravel().round(0))

In [ ]:
mimo = Sequential()
mimo.add(Input(shape=(n_steps, n_features)))
mimo.add(LSTM(32))
mimo.add(Dense(horizon))          # one output per hour of the horizon
mimo.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = mimo.fit(Xm_tr, ym_tr, epochs=50, batch_size=64, validation_split=0.2, callbacks=[es], verbose=0)
# the loss curve - always keep the History object that fit() hands back
plt.figure(figsize=(7, 2.8))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.xlabel("epoch"); plt.ylabel("loss (mse)"); plt.title("multi-output Dense(24): loss curve")
plt.legend(); plt.tight_layout(); plt.show()
print("stopped after", len(history.history["loss"]), "epochs")

mimo_mw = to_mw(mimo.predict(Xm_te, verbose=0))
mae_mimo = np.abs(mimo_mw - to_mw(ym_te)).mean(axis=0)
print("multi-output - MAE at h=1, 6, 12, 24:", mae_mimo[[0, 5, 11, 23]].round(1))

## Error by horizon: watch it fall apart

One number per strategy is not enough here. What matters is **how the error grows as you forecast further out**,
so everything below is reported per horizon: hour 1, hour 2, ... hour 24.

Read the table first. The picture underneath says the same thing.

In [ ]:
# seasonal naive at every horizon: "the same hour yesterday", scored on the same origins
dem = test["Demand"].values
naive = []
for h in range(horizon):
    actual    = dem[n_steps+h      : n_steps+h      + n_orig]
    yesterday = dem[n_steps+h-24   : n_steps+h-24   + n_orig]
    naive.append(mean_absolute_error(actual, yesterday))
naive = np.array(naive)


In [ ]:
# the table, before the picture
rows = {"recursive (frozen wx)": mae_rec_persist,
        "recursive (perfect wx)": mae_rec_actual,
        "multi-output Dense(24)": mae_mimo,
        "seasonal naive": naive}

H = [0, 5, 11, 23]                          # hours 1, 6, 12, 24
board = {}
for name, curve in rows.items():
    board[name] = curve[H].round(1)
board = pd.DataFrame(board, index=["h=1", "h=6", "h=12", "h=24"]).T
board["direct"] = np.nan
for k, h in enumerate(direct_h):
    board.loc["direct (one model per horizon)", "h=" + str(h)] = round(direct_mae[k], 1)
display(board.drop(columns="direct"))

print()
for name, curve in rows.items():
    grew = curve[23] / curve[0]
    print(name.ljust(26), "h=1:", round(curve[0], 1), " h=24:", round(curve[23], 1),
          "  ->", round(grew, 1), "x worse")

**That last column is the whole lesson.** The recursive strategy starts out competitive at one hour and then
degrades fastest, because at hour 24 it has been fed 23 of its own guesses in a row. The multi-output model never
eats its own output, so it degrades more gently. Seasonal naive barely degrades at all - it is always looking a
full day back, whether you ask it for hour 1 or hour 24 - which is exactly why it is such a stubborn baseline.

### The same numbers as a picture

Every line is one strategy, every point is one horizon. The lines that climb left to right are the error falling
apart; the flat grey one is the baseline that does not care how far ahead you ask.

In [ ]:
# the same numbers as a picture
hrs = np.arange(1, horizon+1)
plt.figure(figsize=(9, 4))
plt.plot(hrs, mae_rec_persist, marker=".", label="recursive - weather frozen (honest)")
plt.plot(hrs, mae_rec_actual,  marker=".", label="recursive - perfect-foresight weather (cheat)")
plt.plot(hrs, mae_mimo,        marker=".", label="multi-output Dense(24)")
plt.plot(direct_h, direct_mae, "s", ms=8, label="direct (one model per horizon)")
plt.plot(hrs, naive, "--", color="gray", label="seasonal naive")
plt.xlabel("hours ahead"); plt.ylabel("test MAE (MW)"); plt.title("Three ways to forecast tomorrow"); plt.legend(); plt.show()


In [ ]:
# one origin, the three forecasts and reality
o = 24*200 + 6      # a day in mid-July, forecast made at 6 AM
plt.figure(figsize=(9, 3.5))
plt.plot(hrs, truth_mw[o], "k", lw=2, label="actual")
plt.plot(hrs, rec_persist_mw[o], label="recursive (frozen wx)"); plt.plot(hrs, rec_actual_mw[o], label="recursive (perfect wx)")
plt.plot(hrs, mimo_mw[o], label="multi-output")
plt.xlabel("hours ahead"); plt.ylabel("MW"); plt.title(f"Forecast issued {test.index[o+n_steps-1]}"); plt.legend(); plt.show()

### The same two weeks, forecast twice

**Top panel:** each hour, as forecast one hour earlier. **Bottom panel:** the very same hours, as forecast a full
day earlier. Same data, same models - only the lead time changes.

This is the error-by-horizon table again, drawn as a time series, so you can see *where* the extra error goes
rather than just how big it is.

In [ ]:
# Two weeks in July 2019, picked by DATE. We line each forecast up with the hour it was FOR,
# not the hour it was made - otherwise you are comparing a forecast to the wrong reality.
first_hour = test.index[n_steps]                     # the hour the first forecast is for
start = 24 * 181                                     # rows into the test set
stop  = 24 * 195

fig, ax = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True, sharey=True)

for panel, h in enumerate([1, 24]):
    a = ax[panel]
    rows = np.arange(start, stop)
    when = test.index[n_steps + h - 1 + rows]        # the timestamp each forecast is FOR

    a.plot(when, truth_mw[rows, h-1], color="black", lw=1.6, label="actual")
    a.plot(when, rec_persist_mw[rows, h-1], lw=1.0, label="recursive, weather frozen")
    a.plot(when, rec_actual_mw[rows, h-1], lw=1.0, ls="--", label="recursive, perfect weather")
    a.plot(when, mimo_mw[rows, h-1], lw=1.0, label="multi-output Dense(24)")

    if h == 1:
        a.set_title("forecasts made 1 hour earlier")
    else:
        a.set_title("the same hours, forecast 24 hours earlier")
    a.set_ylabel("MW")

ax[0].legend(ncol=4, fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

**What to look for.** One hour ahead, every strategy sits on the black line. A day ahead, both recursive
forecasts **overshoot the afternoon peaks** - each step's error fed the next one - and the frozen-weather version
overshoots the most. The multi-output model keeps the daily shape and usually sits a little low.

## Where do the future covariates come from?

The recursive strategy forced the question; it applies to *every* strategy that uses covariates at the target hour. Three kinds of column:

| column | at hour +h you have... | what to do |
| :-- | :-- | :-- |
| **calendar / clock** (hour, weekday, holiday) | the exact value - it's a calendar | use it; it's free and it's the strongest covariate after demand itself |
| **weather** (temperature, dew point, humidity) | *not* the actual - only a **forecast** | feed the NWS forecast for hour +h; the forecast's error becomes your error, so validate with *archived forecasts*, not actuals (the "perfect foresight" curve above is the optimistic bound) |
| **the target's own past** (demand) | actuals up to now, then nothing | recursive: your predictions (compounding); direct / multi-output: only real history goes in, the model learned the drift |

The professional pattern is **direct or multi-output with forecast weather** - no fake inputs, and the weather forecast error is explicit. Recursive is fine for a few steps and for models that are expensive to retrain per horizon.

**One more way, for later:** an *encoder-decoder* (seq2seq) RNN reads the window and then *generates* the next 24 values one at a time, trained with **teacher forcing** (the true previous value is fed in during training) - which is exactly the exposure-bias situation, made explicit and usually managed with scheduled sampling. That's the architecture that becomes machine translation in Module 5.

## Save the model and use it again

In [ ]:
mimo.save('A5_Multi_Step_Forecasting_mimo.keras')
reloaded = load_model('A5_Multi_Step_Forecasting_mimo.keras')
print("reloaded model reproduces the forecast:", np.allclose(mimo.predict(Xm_te[:5], verbose=0), reloaded.predict(Xm_te[:5], verbose=0)))

## On your own

- Train all 24 direct models (a loop, ~10 minutes on CPU) and plot the full direct curve. Does it beat multi-output at every horizon?
- Recursive with a **week** of history (`n_steps = 168`). Does a longer window slow the error growth?
- Replace the frozen weather with *yesterday's weather at the same hour* (a poor person's forecast). Where does it land between the two recursive curves?
- Add a `holiday` column (`pandas.tseries.holiday.USFederalHolidayCalendar`) - a calendar covariate you know perfectly in advance.